# MyDigitalTwin — Spotify
**Notebook — Ingestion, exploration, nettoyage → Parquet**

Sources :
- `data/raw/SPOTIFY/StreamingHistory_music_0-7.json` — historique d'écoute (~95k événements)
- `data/raw/SPOTIFY/YourLibrary.json` — titres likés (~1 500 tracks, signal d'identité fort)
- `data/raw/SPOTIFY/Playlist1.json` — 31 playlists personnelles
- `data/raw/SPOTIFY/YourSoundCapsule.json` — snapshots hebdomadaires avec **genres** (unique source de genre)

Outputs :
- `data/parquet/spotify_streams.parquet` — écoutes nettoyées (>30s) avec features temporelles
- `data/parquet/spotify_library.parquet` — titres sauvegardés (identité musicale)
- `data/parquet/spotify_playlists.parquet` — tracks de toutes les playlists
- `data/parquet/spotify_sound_capsule.parquet` — genres par semaine (pour K-Means)

## Objectifs ML
| Axe | Source | Signal |
|---|---|---|
| ALS (Oracle) | streams + library + playlists | poids 1 / 3 / 2 |
| K-Means (Dashboard) | streams + sound capsule | heure, jour, durée, **genre** |

## 0. Initialisation Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType

spark = SparkSession.builder \
    .appName("MyDigitalTwin - Spotify") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

## 1. Streaming History — Ingestion

8 fichiers JSON chronologiques (mars 2025 → mars 2026).  
Chaque entrée = un événement d'écoute avec `endTime`, `artistName`, `trackName`, `msPlayed`.

In [ ]:
STREAMING_FILES  = [
    f"../../data/raw/SPOTIFY/StreamingHistory_music_{i}.json"
    for i in range(8)
]

LIBRARY_PATH      = "../../data/raw/SPOTIFY/YourLibrary.json"
PLAYLIST_PATH     = "../../data/raw/SPOTIFY/Playlist1.json"
SOUND_CAPSULE_PATH = "../../data/raw/SPOTIFY/YourSoundCapsule.json"

OUT_STREAMS       = "../../data/parquet/spotify_streams.parquet"
OUT_LIBRARY       = "../../data/parquet/spotify_library.parquet"
OUT_PLAYLISTS     = "../../data/parquet/spotify_playlists.parquet"
OUT_SOUND_CAPSULE = "../../data/parquet/spotify_sound_capsule.parquet"

# Lecture des 8 fichiers en un seul DataFrame
df_streams_raw = spark.read \
    .option("multiLine", "true") \
    .json(STREAMING_FILES)

print(f"Lignes brutes (toutes écoutes) : {df_streams_raw.count():,}")
df_streams_raw.printSchema()
df_streams_raw.show(5, truncate=50)

## 2. Streaming History — Exploration & Data Quality

In [ ]:
print("=== Valeurs nulles ===")
df_streams_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_streams_raw.columns
]).show()

print("\n=== Période couverte ===")
df_streams_raw.agg(
    F.min("endTime").alias("premier"),
    F.max("endTime").alias("dernier"),
    F.countDistinct("artistName").alias("artistes_distincts"),
    F.countDistinct("trackName").alias("titres_distincts")
).show(truncate=False)

In [ ]:
print("=== Distribution msPlayed (ms) ===")
df_streams_raw.select(
    F.min("msPlayed").alias("min_ms"),
    F.max("msPlayed").alias("max_ms"),
    F.mean("msPlayed").alias("mean_ms"),
    F.percentile_approx("msPlayed", 0.5).alias("median_ms")
).show()

print("\n=== Écoutes < 30s (skip) vs >= 30s ===")
df_streams_raw.groupBy(
    F.when(F.col("msPlayed") < 30000, "< 30s (skip)").otherwise(">= 30s").alias("categorie")
).count().orderBy("categorie").show()

## 3. Streaming History — Nettoyage

Règles appliquées :
- **Filtre >30s** : `msPlayed >= 30 000` — élimine les skips et les pré-écoutes
- **Parse `endTime`** : format `"YYYY-MM-DD HH:MM"` → timestamp Spark
- **Features temporelles** : heure, jour de la semaine, mois, année
- **`minutes_played`** : conversion ms → minutes pour lisibilité

In [ ]:
# Filtre écoutes réelles (>30s)
df_streams = df_streams_raw.filter(F.col("msPlayed") >= 30000)

print(f"Lignes après filtre >30s : {df_streams.count():,}")

# Parse endTime "YYYY-MM-DD HH:MM" → timestamp
df_streams = df_streams.withColumn(
    "listen_ts",
    F.to_timestamp(F.col("endTime"), "yyyy-MM-dd HH:mm")
)

# Features temporelles
df_streams = df_streams \
    .withColumn("listen_year",    F.year("listen_ts")) \
    .withColumn("listen_month",   F.date_format("listen_ts", "yyyy-MM")) \
    .withColumn("listen_hour",    F.hour("listen_ts")) \
    .withColumn("listen_weekday", F.dayofweek("listen_ts")) \
    .withColumn("listen_week",    F.weekofyear("listen_ts")) \
    .withColumn("minutes_played", F.round(F.col("msPlayed") / 60000.0, 2))

# Plage nuit (22h–5h) pour K-Means
df_streams = df_streams.withColumn(
    "is_night",
    F.when((F.col("listen_hour") >= 22) | (F.col("listen_hour") <= 5), True).otherwise(False)
)

# Poids ALS : écoute standard = 1.0
df_streams = df_streams.withColumn("interaction_weight", F.lit(1.0))

df_streams.select(
    "artistName", "trackName", "listen_ts", "listen_hour", "listen_weekday", "minutes_played", "is_night"
).show(8, truncate=40)

## 4. Streaming History — Exploration après nettoyage

In [ ]:
print("=== Top 20 artistes les plus écoutés ===")
df_streams.groupBy("artistName") \
    .agg(
        F.count("*").alias("nb_ecoutes"),
        F.round(F.sum("minutes_played"), 1).alias("total_minutes")
    ) \
    .orderBy(F.desc("nb_ecoutes")) \
    .limit(20) \
    .show(truncate=35)

print("\n=== Top 15 titres les plus écoutés ===")
df_streams.groupBy("artistName", "trackName") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(15) \
    .show(truncate=40)

In [ ]:
print("=== Écoutes par heure de la journée ===")
df_streams.groupBy("listen_hour") \
    .count() \
    .orderBy("listen_hour") \
    .show(24)

print("\n=== Écoutes par jour de la semaine (1=dim, 2=lun, ..., 7=sam) ===")
df_streams.groupBy("listen_weekday") \
    .count() \
    .orderBy("listen_weekday") \
    .show()

print("\n=== % écoutes nocturnes (22h–5h) ===")
total = df_streams.count()
night = df_streams.filter(F.col("is_night")).count()
print(f"Nuit : {night:,} / {total:,} ({night/total*100:.1f}%)")

In [ ]:
print("=== Activité mensuelle ===")
df_streams.groupBy("listen_month") \
    .agg(
        F.count("*").alias("nb_ecoutes"),
        F.round(F.sum("minutes_played"), 0).alias("total_minutes")
    ) \
    .orderBy("listen_month") \
    .show(20)

print("\n=== Semaines les plus actives ===")
df_streams.groupBy("listen_year", "listen_week") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10) \
    .show()

## 5. Schéma final Streaming History & écriture Parquet

In [ ]:
df_streams_final = df_streams.select(
    F.col("artistName"),
    F.col("trackName"),
    F.col("msPlayed"),
    F.col("minutes_played"),
    F.col("listen_ts"),
    F.col("listen_year"),
    F.col("listen_month"),
    F.col("listen_hour"),
    F.col("listen_weekday"),
    F.col("listen_week"),
    F.col("is_night"),
    F.col("interaction_weight")
)

print(f"Lignes finales : {df_streams_final.count():,}")
df_streams_final.printSchema()

df_streams_final.write.mode("overwrite").parquet(OUT_STREAMS)
print(f"\n✓ Parquet écrit : {OUT_STREAMS}")

df_check = spark.read.parquet(OUT_STREAMS)
print(f"✓ Vérification lecture : {df_check.count():,} lignes")
df_check.show(3, truncate=40)

---
## 6. YourLibrary — Ingestion

Les titres sauvegardés = ce qui me représente le plus musicalement.  
Signal ALS fort → **poids 3.0** (vs 1.0 pour les streams ordinaires).

In [ ]:
df_lib_raw = spark.read \
    .option("multiLine", "true") \
    .json(LIBRARY_PATH)

df_lib_raw.printSchema()

In [ ]:
# Explosion du tableau 'tracks'
df_library = df_lib_raw \
    .select(F.explode("tracks").alias("t")) \
    .select(
        F.col("t.artist").alias("artistName"),
        F.col("t.album").alias("albumName"),
        F.col("t.track").alias("trackName"),
        F.col("t.uri").alias("trackUri")
    ) \
    .filter(F.col("trackName").isNotNull()) \
    .withColumn("interaction_weight", F.lit(3.0))  # Signal fort : titre sauvegardé

print(f"Titres dans la bibliothèque : {df_library.count():,}")
df_library.show(10, truncate=45)

## 7. YourLibrary — Exploration

In [ ]:
print("=== Top artistes dans la bibliothèque ===")
df_library.groupBy("artistName") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .show(truncate=35)

print("\n=== Artistes présents dans la bibliothèque mais pas dans les streams ===")
lib_artists   = df_library.select("artistName").distinct()
stream_artists = df_streams.select("artistName").distinct()
only_in_lib = lib_artists.subtract(stream_artists)
print(f"Artistes sauvegardés jamais streamés : {only_in_lib.count():,}")
only_in_lib.limit(15).show(truncate=40)

## 8. YourLibrary — Écriture Parquet

In [ ]:
df_library.write.mode("overwrite").parquet(OUT_LIBRARY)
print(f"✓ Parquet écrit : {OUT_LIBRARY}")

df_check = spark.read.parquet(OUT_LIBRARY)
print(f"✓ Vérification lecture : {df_check.count():,} lignes")
df_check.show(3, truncate=45)

---
## 9. Playlist1 — Ingestion

31 playlists personnelles. Chaque track ajouté manuellement = curation active.  
Signal ALS intermédiaire → **poids 2.0**.

In [ ]:
df_pl_raw = spark.read \
    .option("multiLine", "true") \
    .json(PLAYLIST_PATH)

df_pl_raw.printSchema()

In [ ]:
# Explosion playlists → items → track
df_playlists = df_pl_raw \
    .select(F.explode("playlists").alias("pl")) \
    .select(
        F.col("pl.name").alias("playlistName"),
        F.col("pl.lastModifiedDate").alias("lastModifiedDate"),
        F.explode("pl.items").alias("item")
    ) \
    .select(
        F.col("playlistName"),
        F.col("lastModifiedDate"),
        F.col("item.track.trackName").alias("trackName"),
        F.col("item.track.artistName").alias("artistName"),
        F.col("item.track.albumName").alias("albumName"),
        F.col("item.track.trackUri").alias("trackUri"),
        F.to_date(F.col("item.addedDate"), "yyyy-MM-dd").alias("addedDate")
    ) \
    .filter(F.col("trackName").isNotNull()) \
    .withColumn("interaction_weight", F.lit(2.0))  # Curation manuelle

print(f"Tracks totaux dans les playlists : {df_playlists.count():,}")
df_playlists.show(8, truncate=40)

## 10. Playlists — Exploration

In [ ]:
print("=== Toutes les playlists (nom + nb tracks) ===")
df_playlists.groupBy("playlistName", "lastModifiedDate") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=50)

print("\n=== Top artistes toutes playlists confondues ===")
df_playlists.groupBy("artistName") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .show(truncate=35)

In [ ]:
print("=== Artistes présents dans les 3 sources (streams + library + playlists) ===")
pl_artists  = df_playlists.select("artistName").distinct()
lib_artists = df_library.select("artistName").distinct()
str_artists = df_streams.select("artistName").distinct()

core_artists = pl_artists.intersect(lib_artists).intersect(str_artists)
print(f"Artistes présents partout (noyau dur) : {core_artists.count():,}")
core_artists.limit(20).show(truncate=40)

## 11. Playlists — Écriture Parquet

In [ ]:
df_playlists.write.mode("overwrite").parquet(OUT_PLAYLISTS)
print(f"✓ Parquet écrit : {OUT_PLAYLISTS}")

df_check = spark.read.parquet(OUT_PLAYLISTS)
print(f"✓ Vérification lecture : {df_check.count():,} lignes")
df_check.show(3, truncate=40)

---
## 12. YourSoundCapsule — Ingestion

Snapshots hebdomadaires récents. Seule source qui fournit des **données de genre** — absentes de `StreamingHistory`.  
Chaque entrée = une semaine avec `streamCount`, `secondsPlayed`, `topTracks`, `topArtists`, `topGenres`.

In [ ]:
df_sc_raw = spark.read \
    .option("multiLine", "true") \
    .json(SOUND_CAPSULE_PATH)

df_sc_raw.printSchema()
print(f"Snapshots disponibles : {df_sc_raw.select(F.explode('stats')).count():,}")

In [ ]:
# Explosion stats → une ligne par snapshot hebdomadaire
df_sc = df_sc_raw \
    .select(F.explode("stats").alias("s")) \
    .select(
        F.to_date(F.col("s.date"), "yyyy-MM-dd").alias("snapshot_date"),
        F.col("s.streamCount").alias("stream_count"),
        F.round(F.col("s.secondsPlayed") / 60.0, 1).alias("minutes_played"),
        F.col("s.topGenres").alias("top_genres"),     # array<string>
        F.col("s.topArtists").alias("top_artists"),   # array<struct>
        F.col("s.topTracks").alias("top_tracks")      # array<struct>
    ) \
    .filter(F.col("snapshot_date").isNotNull()) \
    .orderBy("snapshot_date")

print(f"Snapshots après nettoyage : {df_sc.count():,}")
df_sc.select("snapshot_date", "stream_count", "minutes_played", "top_genres").show(10, truncate=60)

## 13. YourSoundCapsule — Exploration

In [ ]:
print("=== Activité hebdomadaire (stream_count + minutes) ===")
df_sc.select("snapshot_date", "stream_count", "minutes_played").show(20)

print("\n=== Genres les plus fréquents (toutes semaines confondues) ===")
df_sc.select(F.explode("top_genres").alias("genre")) \
    .groupBy("genre") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .show(truncate=40)

print("\n=== Top artistes par semaine ===")
df_sc.select(
    F.col("snapshot_date"),
    F.explode("top_artists").alias("a")
).select(
    "snapshot_date",
    F.col("a.artistName").alias("artistName"),
    F.col("a.streamCount").alias("streams_semaine")
).orderBy("snapshot_date", F.desc("streams_semaine")) \
 .show(20, truncate=35)

## 14. YourSoundCapsule — Écriture Parquet

In [ ]:
df_sc.write.mode("overwrite").parquet(OUT_SOUND_CAPSULE)
print(f"✓ Parquet écrit : {OUT_SOUND_CAPSULE}")

df_check = spark.read.parquet(OUT_SOUND_CAPSULE)
print(f"✓ Vérification lecture : {df_check.count():,} lignes")
df_check.select("snapshot_date", "stream_count", "minutes_played", "top_genres").show(3, truncate=60)

---
## 15. Résumé global

| Source | Contenu | Output | Usage ML |
|---|---|---|---|
| `StreamingHistory_music_0-7.json` | ~95k écoutes filtrées >30s | `spotify_streams.parquet` | ALS (poids 1) + K-Means (heure/jour) |
| `YourLibrary.json` | ~1 500 titres sauvegardés | `spotify_library.parquet` | ALS (poids 3) |
| `Playlist1.json` | 31 playlists, curation manuelle | `spotify_playlists.parquet` | ALS (poids 2) |
| `YourSoundCapsule.json` | Snapshots hebdo + genres | `spotify_sound_capsule.parquet` | K-Means (genre par semaine) |

**Prochaines étapes :**
- **Notebook ALS** : unifier les 3 sources interactions → matrice `(user, track_id, weight)` → `interactions.parquet`
- **Notebook K-Means** : combiner `listen_hour`, `listen_weekday`, `is_night`, `minutes_played` (streams) + `top_genres` (sound capsule) → `activity_vectors.parquet`

In [ ]:
spark.stop()